In [ ]:
# Label Studio test

> Labelstudio related word

In [ ]:
#| default_exp label_studio

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from label_studio_sdk import Client
import os
from pathlib import Path
from tqdm.auto import tqdm
import requests
from PIL import Image
import numpy as np

In [ ]:
from cv_tools.core import *

In [ ]:
import uuid

In [ ]:
LBL_STD_API=os.getenv('LBL_STUDIO_API')

In [ ]:

# Replace 'localhost' with your Label Studio hostname
LS_HOST = 'http://localhost:8080'
API_KEY = LBL_STD_API

client = Client(url=LS_HOST, api_key=API_KEY)

In [ ]:
project_id = 2
project = client.get_project(project_id)

In [ ]:
im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/images')
msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/masks')

In [ ]:
im_ = im_path.ls()[0]
project.import_tasks(im_)

In [ ]:
model_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/models/time_11_18_24_val_frGrnd0.9470_epoch_185.h5')
model_path.relative_to(main_path)

In [ ]:
tasks_to_import = [
    {
        'data': {
            'image': im_.as_posix(),
            #'image_mask': f'{msk_path.ls()[0]}'
        },
        # Add any additional fields or metadata here
    },
]

# Assuming 'project' is your Label Studio project object
project.import_tasks(tasks_to_import) #preannotated_from_fields=['image_mask'])

In [ ]:
project.import_tasks??

In [ ]:
fns = get_name_(im_path.ls())

In [ ]:
main_path = Path(r'/home/user/Schreibtisch/projects/data')
image_path.relative_to(main_path)

In [ ]:
from label_studio_converter.brush import encode_rle, image2annotation
import json

In [ ]:
im_

In [ ]:
msk_s = msk_path.ls()[0]

In [ ]:
def image2rle(path):
    """Convert mask image (jpg, png) to RLE

    1. Read image as grayscale
    2. Flatten to 1d array
    3. Threshold > 128
    4. Encode

    :param path: path to image with mask (jpg, png), this image will be thresholded with values > 128 to obtain mask,
                 so you can mark background as black and foreground as white
    :return: list of ints in RLE format
    """
    with Image.open(path).convert('L') as image:
        mask = np.array((np.array(image) > 128) * 255, dtype=np.uint8)
        array = mask.ravel()
        array = np.repeat(array, 4)
        rle = encode_rle(array)
        return rle, image.size[0], image.size[1]

In [ ]:
def image2annotation(
    path,
    label_name,
    from_name,
    to_name,
    ground_truth=False,
    model_version=None,
    score=None,
):
    """Convert image with mask to brush RLE annotation

    :param path: path to image with mask (jpg, png), this image will be thresholded with values > 128 to obtain mask,
                 so you can mark background as black and foreground as white
    :param label_name: label name from labeling config (<Label>)
    :param from_name: brush tag name (<BrushLabels>)
    :param to_name: image tag name (<Image>)
    :param ground_truth: ground truth annotation true/false
    :param model_version: any string, only for predictions
    :param score: model score as float, only for predictions

    :return: dict with Label Studio Annotation or Prediction (Pre-annotation)
    """
    rle, width, height = image2rle(path)
    result = {
        "result": [
            {
                "id": str(uuid.uuid4())[0:8],
                "type": "brushlabels",
                "value": {"rle": rle, "format": "rle", "brushlabels": [label_name]},
                "origin": "manual",
                "to_name": to_name,
                "from_name": from_name,
                "image_rotation": 0,
                "original_width": width,
                "original_height": height,
            }
        ],
    }

    # prediction
    if model_version:
        result["model_version"] = model_version
        result["score"] = score

    # annotation
    else:
        result["ground_truth"] = ground_truth

    return result

https://github.com/HumanSignal/label-studio-converter/blob/master/label_studio_converter/brush.py#L361

In [ ]:
project.create_annotation(
    task_id=3, 
    annotation=[res])

In [ ]:
res = image2annotation(
    im_,
    label_name='AirPlane',
    from_name='BrushLabels',
    to_name='Image',

)

In [ ]:
def test_image2annotation():
    """
    Import from png to LS annotation with RLE values
    """
    annotation = image2annotation(
        #os.path.abspath(os.path.dirname(__file__)) + '/data/test_brush/test.png',
        msk_s.as_posix(),
        label_name='Airplane',
        from_name='tag',
        to_name='image',
        model_version='v1',
        score=0.5,
    )

    # prepare Label Studio Task
    task = {
        'data': {'image': im_.as_posix()},
        'predictions': [annotation],
    }

    """ You can import this `task.json` to the Label Studio project with this labeling config:

    <View>
      <Image name="image" value="$image" zoom="true"/>
      <BrushLabels name="tag" toName="image">
        <Label value="Airplane" background="rgba(255, 0, 0, 0.7)"/>
        <Label value="Car" background="rgba(0, 0, 255, 0.7)"/>
      </BrushLabels>
    </View>

    """
    json.dump(task, open('task.json', 'w'))
    assert True

In [ ]:
#| export
def image2rle(path):
    """Convert mask image (jpg, png) to RLE

    1. Read image as grayscale
    2. Flatten to 1d array
    3. Threshold > 128
    4. Encode

    :param path: path to image with mask (jpg, png), this image will be thresholded with values > 128 to obtain mask,
                 so you can mark background as black and foreground as white
    :return: list of ints in RLE format
    """
    with Image.open(path).convert('L') as image:
        mask = np.array((np.array(image) > 128) * 255, dtype=np.uint8)
        array = mask.ravel()
        array = np.repeat(array, 4)
        rle = encode_rle(array)
        return rle, image.size[0], image.size[1]

In [ ]:
def image2annotation(
    path,
    label_name,
    from_name,
    to_name,
    ground_truth=False,
    model_version=None,
    score=None,
):
    """Convert image with mask to brush RLE annotation

    :param path: path to image with mask (jpg, png), this image will be thresholded with values > 128 to obtain mask,
                 so you can mark background as black and foreground as white
    :param label_name: label name from labeling config (<Label>)
    :param from_name: brush tag name (<BrushLabels>)
    :param to_name: image tag name (<Image>)
    :param ground_truth: ground truth annotation true/false
    :param model_version: any string, only for predictions
    :param score: model score as float, only for predictions

    :return: dict with Label Studio Annotation or Prediction (Pre-annotation)
    """
    rle, width, height = image2rle(path)
    result = {
        "result": [
            {
                "id": str(uuid.uuid4())[0:8],
                "type": "brushlabels",
                "value": {"rle": rle, "format": "rle", "brushlabels": [label_name]},
                "origin": "manual",
                "to_name": to_name,
                "from_name": from_name,
                "image_rotation": 0,
                "original_width": width,
                "original_height": height,
            }
        ],
    }

    # prediction
    if model_version:
        result["model_version"] = model_version
        result["score"] = score

    # annotation
    else:
        result["ground_truth"] = ground_truth

    return result

In [ ]:
test_image2annotation()

In [ ]:

for i in tqdm(fns, total=len(fns)):
    image_path = Path(im_path, i)
    mask_path = Path(msk_path, i)
    headers = {
        'Authorization': f'Token {LBL_STD_API}',
        'Content-Type': 'application/json',
    }

    data = {
        "data": {
            "image": f'http://localhost:8000/{image_path.relative_to(main_path)}',
            "image_mask": f'http://localhost:8000/{mask_path.relative_to(main_path)}',
        }
    }
    response = requests.post(
        f'{LS_HOST}/api/projects/{project_id}/tasks/bulk/', 
        json=[data], headers=headers)
    if response.status_code == 201:
        print("Task created successfully.")
    else:
        print(f"Failed to create task: {response.text}")

In [ ]:
for i in tqdm(fns, total=len(fns)):
    image_path = Path(im_path, i)
    mask_path = Path(msk_path, i)
    tasks = client.create_task(
        project_id=project_id,
        data={'image': str(image_path), 
              'image_mask': str(mask_path)
             }
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('11_label_studio_polygons.ipynb')